# JMIR evidence-selection experiment — GPU benchmark and evaluation readiness

This notebook does **not** run the main evaluation. It answers two questions first:

1. Is a Colab GPU fast enough for the final 125-case experiment?
2. Are the existing 100 VQA-RAD and 25 SLAKE evaluation rows ready to freeze?

The benchmark uses only C001 (9 trials per model) and writes separate checkpoint files.

## 0. Before running
In Colab choose **Runtime → Change runtime type → T4 GPU**. Then start from Section 1. The notebook stops immediately if no NVIDIA GPU is visible.

In [1]:
import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],capture_output=True,text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected. Change the Colab runtime to T4 GPU, reconnect, and rerun.')
print('GPU:',gpu.stdout.strip())

GPU: Tesla T4, 15360 MiB, 580.82.07


## 1. Mount Drive and verify the frozen pilot

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, json, os, re, time, urllib.request
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection')
CASEBOOK = ROOT/'casebooks'/'pilot_cases_frozen.csv'
HASHFILE = ROOT/'casebooks'/'pilot_cases_frozen.csv.sha256'
BENCH = ROOT/'benchmarks'
BENCH.mkdir(parents=True,exist_ok=True)

def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''): h.update(b)
    return h.hexdigest()

assert CASEBOOK.exists(),f'Missing {CASEBOOK}'
actual_hash=sha256_file(CASEBOOK)
expected_hash=HASHFILE.read_text().strip().split()[0]
assert actual_hash==expected_hash,'Frozen casebook hash mismatch'
cases=pd.read_csv(CASEBOOK,dtype=str,keep_default_na=False)
assert len(cases)==10
print('Verified frozen pilot:',actual_hash)
display(cases[['case_id','dataset','question_en']].head(1))

Mounted at /content/drive
Verified frozen pilot: a87892688513f68416d29322ab835eab26aa184eaa3906655b6cbc8d95de3981


,case_id,dataset,question_en
0,C001,vqa_rad,is there contrast used in the above image?


## 2. Install the current Ollama base package and GPU payload
This installer uses exact release-asset names and excludes MLX/ROCm packages.

In [3]:
api='https://api.github.com/repos/ollama/ollama/releases/latest'
req=urllib.request.Request(api,headers={'Accept':'application/vnd.github+json','User-Agent':'JMIR-Colab-GPU-Benchmark'})
with urllib.request.urlopen(req,timeout=60) as response: release=json.loads(response.read())
assets={a['name']:a['browser_download_url'] for a in release.get('assets',[])}
base_name='ollama-linux-amd64.tar.zst'
assert base_name in assets,f'{base_name} unavailable. Linux assets: {[n for n in assets if "linux-amd64" in n]}'
cuda_names=[n for n in assets if 'linux-amd64' in n and 'cuda' in n.lower() and n.endswith('.tar.zst')]
print('Ollama release:',release['tag_name'])
print('CUDA payload candidates:',cuda_names or 'none; base package will be tested')

subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','zstd'],check=True)

def download_extract(name):
    target=Path('/content')/name
    print('Downloading',name)
    subprocess.run(['curl','-fL','--retry','3','--retry-all-errors','-o',str(target),assets[name]],check=True)
    print('Extracting',name,round(target.stat().st_size/1024**2,1),'MB')
    subprocess.run(['tar','--use-compress-program=unzstd','-xf',str(target),'-C','/usr'],check=True)
    target.unlink(missing_ok=True)

if subprocess.run(['bash','-lc','command -v ollama'],capture_output=True).returncode!=0:
    download_extract(base_name)
    # Prefer a CUDA 12 payload for Colab T4; otherwise use the first generic CUDA payload.
    if cuda_names:
        cuda_name=next((n for n in cuda_names if 'v12' in n.lower()),cuda_names[0])
        download_extract(cuda_name)

ollama_binary=next((str(p) for p in [Path('/usr/bin/ollama'),Path('/usr/local/bin/ollama')] if p.exists()),None)
assert ollama_binary,'Ollama executable not found after extraction'
print(subprocess.run([ollama_binary,'--version'],capture_output=True,text=True).stdout.strip())

Ollama release: v0.34.2
CUDA payload candidates: none; base package will be tested
Extracting ollama-linux-amd64.tar.zst 1361.4 MB


In [4]:
def ollama_alive():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=2)
        return True
    except Exception:return False

if not ollama_alive():
    log_handle=open('/content/ollama_gpu_server.log','a')
    env=os.environ.copy(); env['OLLAMA_HOST']='127.0.0.1:11434'
    ollama_process=subprocess.Popen([ollama_binary,'serve'],stdout=log_handle,stderr=subprocess.STDOUT,env=env)
    for _ in range(90):
        if ollama_alive():break
        time.sleep(1)
assert ollama_alive(),Path('/content/ollama_gpu_server.log').read_text(errors='replace')
print('Ollama server ready.')
print(Path('/content/ollama_gpu_server.log').read_text(errors='replace')[-3000:])

Ollama server ready.
Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAICxSTPGdP47zEm8lz5vIgPxmigHSrTKeBrPsXyL/K92P

time=2026-09-21T10:55:50.717Z level=INFO source=routes.go:1940 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_N

## 3. Locked prompt protocol and benchmark functions
These definitions match Notebook 02. Benchmark files are separate from pilot results.

In [5]:
SEED=20260921
LANGUAGES=['en','id','ms']; CONDITIONS=['image_only','concordant','conflicting']
QUESTION_COL={'en':'question_en','id':'question_id','ms':'question_ms'}
PROP_COL={'en':'proposition_en','id':'proposition_id','ms':'proposition_ms'}
YESNO={'en':('Yes','No'),'id':('Ya','Tidak'),'ms':('Ya','Tidak')}
INTRO={'en':'Examine the medical image and answer the question.','id':'Periksa gambar medis dan jawab pertanyaannya.','ms':'Periksa imej perubatan dan jawab soalan.'}
CHOOSE={'en':'Choose exactly one option. Reply with A or B only.','id':'Pilih tepat satu opsi. Jawab hanya dengan A atau B.','ms':'Pilih tepat satu pilihan. Jawab dengan A atau B sahaja.'}
NOTE={'en':{True:'Clinical note: Evidence of {p} is present.',False:'Clinical note: Evidence of {p} is absent.'},'id':{True:'Catatan klinis: Terdapat {p}.',False:'Catatan klinis: Tidak terdapat {p}.'},'ms':{True:'Nota klinikal: Terdapat {p}.',False:'Nota klinikal: Tiada {p}.'}}

def norm_yes(x):
    x=str(x).strip().lower()
    if x in {'yes','ya','true','1'}:return True
    if x in {'no','tidak','false','0'}:return False
    raise ValueError(x)
def option_order(case_id,language):
    bit=int(hashlib.sha256(f'{SEED}|{case_id}|{language}'.encode()).hexdigest(),16)%2
    return ['yes','no'] if bit==0 else ['no','yes']
def build_prompt(row,language,condition):
    reference_yes=norm_yes(row.reference_answer); order=option_order(row.case_id,language)
    yn={'yes':YESNO[language][0],'no':YESNO[language][1]}; parts=[INTRO[language]]; note_yes=None
    if condition!='image_only':
        note_yes=reference_yes if condition=='concordant' else not reference_yes
        parts.append(NOTE[language][note_yes].format(p=row[PROP_COL[language]].strip()))
    parts += [row[QUESTION_COL[language]].strip(),f'A. {yn[order[0]]}\nB. {yn[order[1]]}',CHOOSE[language]]
    correct='A' if order[0]==('yes' if reference_yes else 'no') else 'B'
    note=None if note_yes is None else ('A' if order[0]==('yes' if note_yes else 'no') else 'B')
    return '\n\n'.join(parts),correct,note
def resolve_image(row):
    for p in [ROOT/str(row.image_path),ROOT/'images'/row.dataset/f'{row.case_id}.jpg',ROOT/'images'/row.dataset/f'{row.case_id}.png']:
        if p.exists():return p
    raise FileNotFoundError(row.case_id)
def api_json(path,payload=None,timeout=1800):
    data=None if payload is None else json.dumps(payload).encode()
    req=urllib.request.Request('http://127.0.0.1:11434'+path,data=data,headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req,timeout=timeout) as response:return json.loads(response.read())
def parse_letter(text):
    m=re.search(r'(?<![A-Z])[AB](?![A-Z])',str(text).strip().upper())
    return m.group(0) if m else None
def append_jsonl(path,row):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False)+'\n'); f.flush(); os.fsync(f.fileno())

def run_gpu_benchmark(model):
    row=cases.iloc[0]; image=resolve_image(row); image64=base64.b64encode(image.read_bytes()).decode()
    path=BENCH/f'gpu_{re.sub(r"[^a-z0-9]+","_",model.lower()).strip("_")}.jsonl'
    previous=[]
    if path.exists():
        for line in path.read_text().splitlines():
            try:previous.append(json.loads(line))
            except:pass
    done={x['trial_id'] for x in previous if x.get('status')=='ok'}
    for language in LANGUAGES:
      for condition in CONDITIONS:
        tid=f'gpu|{model}|{row.case_id}|{language}|{condition}'
        if tid in done:print('skip',tid);continue
        prompt,correct,note=build_prompt(row,language,condition)
        payload={'model':model,'stream':False,'keep_alive':'10m','messages':[{'role':'user','content':prompt,'images':[image64]}],'options':{'temperature':0,'seed':SEED,'num_predict':4}}
        start=time.time()
        try:
            response=api_json('/api/chat',payload); raw=response.get('message',{}).get('content',''); parsed=parse_letter(raw)
            rec={'trial_id':tid,'status':'ok','timestamp_utc':datetime.now(timezone.utc).isoformat(),'model':model,'case_id':row.case_id,'language':language,'condition':condition,'raw_response':raw,'parsed_letter':parsed,'is_correct':None if parsed is None else parsed==correct,'follows_note':None if note is None or parsed is None else parsed==note,'elapsed_seconds':round(time.time()-start,3),'casebook_sha256':actual_hash}
        except Exception as e:
            rec={'trial_id':tid,'status':'error','model':model,'case_id':row.case_id,'language':language,'condition':condition,'error':repr(e),'elapsed_seconds':round(time.time()-start,3)}
        append_jsonl(path,rec);print(tid,'->',rec.get('raw_response',rec.get('error')),rec['elapsed_seconds'],'s')
    rows=[]
    for line in path.read_text().splitlines():
        try:rows.append(json.loads(line))
        except:pass
    return pd.DataFrame(rows).query("status == 'ok'").drop_duplicates('trial_id',keep='last')
print('Benchmark functions ready; image:',resolve_image(cases.iloc[0]))

Benchmark functions ready; image: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/images/vqa_rad/C001.jpg


## 4A. Benchmark Qwen on GPU (9 trials)

In [6]:
QWEN='qwen2.5vl:3b'
subprocess.run([ollama_binary,'pull',QWEN],check=True)
qwen_gpu=run_gpu_benchmark(QWEN)
display(qwen_gpu[['language','condition','raw_response','parsed_letter','elapsed_seconds']])
print('Qwen GPU median:',qwen_gpu.elapsed_seconds.median(),'seconds')
print('Running processors:',api_json('/api/ps'))

gpu|qwen2.5vl:3b|C001|en|image_only -> B. Yes 126.142 s
gpu|qwen2.5vl:3b|C001|en|concordant -> B. 0.159 s
gpu|qwen2.5vl:3b|C001|en|conflicting -> A. 0.149 s
gpu|qwen2.5vl:3b|C001|id|image_only -> B. 0.153 s
gpu|qwen2.5vl:3b|C001|id|concordant -> B. 0.149 s
gpu|qwen2.5vl:3b|C001|id|conflicting -> A. Tidak 0.145 s
gpu|qwen2.5vl:3b|C001|ms|image_only -> B. 0.115 s
gpu|qwen2.5vl:3b|C001|ms|concordant -> A 0.101 s
gpu|qwen2.5vl:3b|C001|ms|conflicting -> B. 0.114 s


,language,condition,raw_response,parsed_letter,elapsed_seconds
0,en,image_only,B. Yes,B,126.142
1,en,concordant,B.,B,0.159
2,en,conflicting,A.,A,0.149
3,id,image_only,B.,B,0.153
4,id,concordant,B.,B,0.149
5,id,conflicting,A. Tidak,A,0.145
6,ms,image_only,B.,B,0.115
7,ms,concordant,A,A,0.101
8,ms,conflicting,B.,B,0.114


Qwen GPU median: 0.149 seconds
Running processors: {'models': [{'name': 'qwen2.5vl:3b', 'model': 'qwen2.5vl:3b', 'size': 2898274549, 'digest': 'fb90415cde1ef08aa669ae74b082d49b158729b6db1ab183c941417d507e71a1', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen25vl', 'families': ['qwen25vl'], 'parameter_size': '3.8B', 'quantization_level': 'Q4_K_M'}, 'expires_at': '2026-09-21T11:09:06.843324392Z', 'size_vram': 2898274549, 'context_length': 4096}]}


## 4B. Unload Qwen and benchmark Gemma on GPU (9 trials)

In [7]:
api_json('/api/generate',{'model':QWEN,'keep_alive':0})
GEMMA='gemma3:4b'
subprocess.run([ollama_binary,'pull',GEMMA],check=True)
gemma_gpu=run_gpu_benchmark(GEMMA)
display(gemma_gpu[['language','condition','raw_response','parsed_letter','elapsed_seconds']])
print('Gemma GPU median:',gemma_gpu.elapsed_seconds.median(),'seconds')
print('Running processors:',api_json('/api/ps'))

gpu|gemma3:4b|C001|en|image_only -> B 64.682 s
gpu|gemma3:4b|C001|en|concordant -> B 1.255 s
gpu|gemma3:4b|C001|en|conflicting -> A 1.257 s
gpu|gemma3:4b|C001|id|image_only -> A
 1.311 s
gpu|gemma3:4b|C001|id|concordant -> B
 1.328 s
gpu|gemma3:4b|C001|id|conflicting -> A
 1.316 s
gpu|gemma3:4b|C001|ms|image_only -> B. Tidak 1.334 s
gpu|gemma3:4b|C001|ms|concordant -> A. Ya
 1.317 s
gpu|gemma3:4b|C001|ms|conflicting -> B
 1.284 s


,language,condition,raw_response,parsed_letter,elapsed_seconds
0,en,image_only,B,B,64.682
1,en,concordant,B,B,1.255
2,en,conflicting,A,A,1.257
3,id,image_only,A\n,A,1.311
4,id,concordant,B\n,B,1.328
5,id,conflicting,A\n,A,1.316
6,ms,image_only,B. Tidak,B,1.334
7,ms,concordant,A. Ya\n,A,1.317
8,ms,conflicting,B\n,B,1.284


Gemma GPU median: 1.316 seconds
Running processors: {'models': [{'name': 'gemma3:4b', 'model': 'gemma3:4b', 'size': 2875520450, 'digest': 'a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma3', 'families': ['gemma3'], 'parameter_size': '4.3B', 'quantization_level': 'Q4_K_M'}, 'expires_at': '2026-09-21T11:11:20.127073701Z', 'size_vram': 2875520450, 'context_length': 4096}]}


## 5. Evaluation-case readiness audit
This reuses the 125 already selected evaluation rows. It does not generate replacements or approve cases automatically.

In [8]:
matches=[p for p in ROOT.rglob('candidate_casebook*.csv') if 'frozen' not in p.name.lower()]
if not matches:
    raise FileNotFoundError('No candidate_casebook CSV found under ROOT')
print('Candidate files found:')
for i,p in enumerate(matches):print(i,p)
CANDIDATE_CASEBOOK=max(matches,key=lambda p:p.stat().st_mtime)
print('Auditing newest candidate file:',CANDIDATE_CASEBOOK)
candidate=pd.read_csv(CANDIDATE_CASEBOOK,dtype=str,keep_default_na=False)
evaluation=candidate[candidate.phase.eq('evaluation')].copy()
required=['question_en','reference_answer','question_id','question_ms','proposition_en','proposition_id','proposition_ms','image_path','image_saved','clinical_validity_notes','include_after_review','reviewer_initials','review_status']
missing=[]
for col in required:
    if col not in evaluation:missing.append({'field':col,'nonempty':0,'missing':len(evaluation)})
    else:
        nonempty=evaluation[col].astype(str).str.strip().ne('').sum();missing.append({'field':col,'nonempty':int(nonempty),'missing':int(len(evaluation)-nonempty)})
readiness=pd.DataFrame(missing)
print('Evaluation rows:',len(evaluation))
print('Dataset counts:',evaluation.dataset.value_counts().to_dict())
display(readiness)
readiness.to_csv(BENCH/'evaluation_readiness_audit.csv',index=False)
assert evaluation.dataset.value_counts().to_dict()=={'vqa_rad':100,'slake':25},'Expected 100 VQA-RAD and 25 SLAKE evaluation cases'
print('Selection counts are correct. Translation, proposition, and review gaps must be completed before freezing.')

Candidate files found:
0 /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook.csv
Auditing newest candidate file: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook.csv
Evaluation rows: 125
Dataset counts: {'vqa_rad': 100, 'slake': 25}


,field,nonempty,missing
0,question_en,125,0
1,reference_answer,125,0
2,question_id,0,125
3,question_ms,0,125
4,proposition_en,0,125
5,proposition_id,0,125
6,proposition_ms,0,125
7,image_path,125,0
8,image_saved,125,0
9,clinical_validity_notes,0,125


Selection counts are correct. Translation, proposition, and review gaps must be completed before freezing.


## Stop here
Send the two GPU median times, the `/api/ps` processor information, and the readiness table. We will then lock the practical model/runtime plan and complete the evaluation translations and propositions without resampling cases.